In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/biohub-cell-tracking-during-development/sample_submission.csv
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/zarr.json
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/zarr.json
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/7/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/47/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/17/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/81/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/19/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845f.zarr/0/c/22/0/0/0
/kaggle/input/competitions/biohub-cell-tracking-during-development/test/44b6_0b24845

In [2]:
# ============================================================================
# BIOHUB — ZEBRAFISH CELL TRACKING | COMPLETE OFFLINE KAGGLE PIPELINE
# ============================================================================
# Purpose
#   Detect cell centers in 3D time-lapse microscopy, link detections into a
#   temporal graph, recover short gaps, conservatively infer divisions, and
#   write the exact competition submission.csv schema.
#
# Runtime contract
#   * Offline / no network calls
#   * Kaggle notebook-compatible
#   * Works with common Zarr v2/v3 array/group layouts
#   * Writes /kaggle/working/submission.csv
#
# Competition-grounded constants
#   * Image shape: T,Z,Y,X (with singleton-channel tolerance)
#   * Physical voxel scale: (1.625, 0.40625, 0.40625) um / voxel
#   * Node matching gate: 7 um
#   * Submission: node + edge rows, id is consecutive throwaway index
#
# Governance layer
#   * node_id >= 1
#   * -1 is used only for empty submission fields
#   * detection/tracking computations remain numeric floating-point where
#     required by image processing; the integer constraint is enforced at the
#     coordinate / identifier / output boundary.
# ============================================================================

import os
import time
import json
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

# Kaggle's mounted competition directory can differ between notebook versions,
# so discovery below checks several safe locations instead of assuming one
# brittle path.
INPUT_CANDIDATES = [
    Path("/kaggle/input/biohub-cell-tracking-during-development"),
    Path("/kaggle/input/competitions/biohub-cell-tracking-during-development"),
    Path("/kaggle/input"),
]
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Physical scale: z is sampled much more coarsely than y/x.
SCALE = np.asarray([1.625, 0.40625, 0.40625], dtype=np.float32)

# Detection bank. Multiple scales are evaluated and merged.
SIGMAS = (
    (0.8, 1.6, 1.6),
    (1.2, 2.5, 2.5),
    (1.8, 3.5, 3.5),
)
PEAK_WINDOWS = (
    (5, 9, 9),
    (5, 11, 11),
    (7, 13, 13),
)
MIN_PEAK_SEPARATION_UM = 2.0
MAX_DETECTIONS_PER_FRAME = 6000

# Adaptive thresholding. The detector lowers the threshold in steps if a frame
# is implausibly sparse, but never below the configured floor.
THRESHOLD_QUANTILES = (0.9995, 0.998, 0.995, 0.99)
MIN_RELATIVE_THRESHOLD = 0.055

# Tracking.
MAX_LINK_UM = 7.0
MAX_GAP = 2
MOTION_WEIGHT = 0.35
VELOCITY_CLIP_UM_PER_FRAME = 7.0

# Division detection is deliberately conservative: edge-Jaccard is the main
# score component, so false division edges are more harmful than missed forks.
ENABLE_DIVISIONS = True
DIVISION_MAX_UM = 10.0
DIVISION_MIN_SEPARATION_UM = 1.0
DIVISION_MAX_DAUGHTERS = 2

# Safety / output.
MIN_NODES_PER_DATASET = 1
SAVE_QC_JSON = True

# ---------------------------------------------------------------------------
# INPUT DISCOVERY
# ---------------------------------------------------------------------------

def _is_zarr_path(p: Path) -> bool:
    return p.name.endswith(".zarr") or p.suffix == ".zarr" or p.is_dir()


def find_competition_root():
    """Locate the mounted BioHub competition directory without web access."""
    # First pass: direct candidates.
    for root in INPUT_CANDIDATES:
        if not root.exists() or not root.is_dir():
            continue
        test = root / "test"
        if test.exists() and test.is_dir():
            return root

    # Second pass: locate a directory containing test/ with zarr-like children.
    for base in (Path("/kaggle/input"),):
        if not base.exists():
            continue
        for child in sorted(base.iterdir()):
            if not child.is_dir():
                continue
            test = child / "test"
            if not test.is_dir():
                continue
            try:
                members = [p for p in test.iterdir() if _is_zarr_path(p)]
            except Exception:
                members = []
            if members:
                return child

    raise FileNotFoundError(
        "Could not locate the BioHub competition input. Expected a mounted "
        "competition directory containing test/*.zarr."
    )


def dataset_name(path: Path) -> str:
    return path.name[:-5] if path.name.endswith(".zarr") else path.name


def find_test_datasets(root: Path):
    test_dir = root / "test"
    if not test_dir.exists():
        raise FileNotFoundError(f"Missing test directory: {test_dir}")

    paths = sorted(
        p for p in test_dir.iterdir()
        if p.name.endswith(".zarr") or p.is_dir()
    )
    if not paths:
        raise RuntimeError(f"No Zarr test datasets found in {test_dir}")
    return paths

# ---------------------------------------------------------------------------
# ZARR I/O
# ---------------------------------------------------------------------------

def open_zarr_array(path: Path):
    """Open common Zarr v2/v3 direct-array and group layouts."""
    try:
        import zarr
    except Exception as exc:
        raise RuntimeError(
            "zarr is not installed in this Kaggle environment. Attach/use a "
            "Kaggle runtime that includes zarr before running the notebook."
        ) from exc

    root = zarr.open(str(path), mode="r")

    if hasattr(root, "shape"):
        return root

    # Common array names.
    for key in ("0", "data", "image", "0/0"):
        try:
            obj = root[key]
            if hasattr(obj, "shape"):
                return obj
        except Exception:
            pass

    # Search descendants one level at a time, preferring numeric keys.
    try:
        keys = list(root.keys())
        keys.sort(key=lambda k: (not str(k).isdigit(), str(k)))
        for key in keys:
            obj = root[key]
            if hasattr(obj, "shape"):
                return obj
            try:
                subkeys = list(obj.keys())
                subkeys.sort(key=lambda k: (not str(k).isdigit(), str(k)))
                for sk in subkeys:
                    sub = obj[sk]
                    if hasattr(sub, "shape"):
                        return sub
            except Exception:
                pass
    except Exception:
        pass

    raise RuntimeError(f"Could not find an image array inside {path}")


def make_array_adapter(arr):
    """Normalize supported input shapes to an object exposing T,Z,Y,X."""
    shape = tuple(int(v) for v in arr.shape)

    if len(shape) == 4:
        # T,Z,Y,X
        return arr

    if len(shape) == 5:
        # T,C,Z,Y,X with singleton channel.
        if shape[1] == 1:
            class ChannelAdapter:
                shape = (shape[0], shape[2], shape[3], shape[4])
                def __getitem__(self, key):
                    return arr[key, 0]
            return ChannelAdapter()

        # Some OME layouts can put a singleton dimension at the end.
        if shape[-1] == 1:
            class LastChannelAdapter:
                shape = (shape[0], shape[1], shape[2], shape[3])
                def __getitem__(self, key):
                    return arr[key][..., 0]
            return LastChannelAdapter()

    raise ValueError(f"Unsupported array shape {shape}; expected T,Z,Y,X")


def read_frame(arr, t: int) -> np.ndarray:
    frame = np.asarray(arr[t])
    while frame.ndim > 3 and frame.shape[0] == 1:
        frame = frame[0]
    while frame.ndim > 3 and frame.shape[-1] == 1:
        frame = frame[..., 0]
    if frame.ndim != 3:
        raise ValueError(f"Frame {t} is not 3D: shape={frame.shape}")
    return frame.astype(np.float32, copy=False)

# ---------------------------------------------------------------------------
# IMAGE NORMALIZATION / DETECTION
# ---------------------------------------------------------------------------

def robust_normalize(frame: np.ndarray) -> np.ndarray:
    x = frame.astype(np.float32, copy=False)
    if x.size == 0:
        return x
    lo, hi = np.percentile(x, [1.0, 99.9])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(x, dtype=np.float32)
    x = np.clip((x - lo) / (hi - lo), 0.0, 1.0)
    return x.astype(np.float32, copy=False)


def _local_maxima(response: np.ndarray, window):
    mx = maximum_filter(response, size=window, mode="nearest")
    return response == mx


def _physical_nms(coords, scores, min_sep_um, max_keep):
    if len(coords) == 0:
        return np.empty((0, 3), dtype=np.int32)

    order = np.argsort(scores)[::-1]
    kept = []
    kept_phys = []
    sep2 = float(min_sep_um) ** 2

    for idx in order:
        c = coords[idx].astype(np.int32, copy=False)
        p = c.astype(np.float32) * SCALE
        if kept_phys:
            a = np.asarray(kept_phys, dtype=np.float32)
            d2 = np.sum((a - p) ** 2, axis=1)
            if np.any(d2 < sep2):
                continue
        kept.append(c)
        kept_phys.append(p)
        if len(kept) >= max_keep:
            break

    return np.asarray(kept, dtype=np.int32)


def detect_frame(frame: np.ndarray) -> np.ndarray:
    """Multi-scale 3D local-max detector returning integer z,y,x centroids."""
    x = robust_normalize(frame)
    if x.size == 0 or float(x.max()) <= 0:
        return np.empty((0, 3), dtype=np.int32)

    candidates = []
    scores = []

    # Multi-scale DoG bank.
    for sigma, window in zip(SIGMAS, PEAK_WINDOWS):
        g1 = gaussian_filter(x, sigma=sigma, mode="nearest")
        sigma2 = tuple(float(s) * 1.7 for s in sigma)
        g2 = gaussian_filter(x, sigma=sigma2, mode="nearest")
        dog = np.maximum(g1 - g2, 0.0)
        if float(dog.max()) <= 0:
            continue

        mx = float(dog.max())
        base_mask = _local_maxima(dog, window)
        local_scores = dog[base_mask]
        if local_scores.size == 0:
            continue

        # Try progressively lower quantiles only when necessary.
        q_candidates = [float(np.quantile(dog, q)) for q in THRESHOLD_QUANTILES]
        q_candidates = [max(q, MIN_RELATIVE_THRESHOLD * mx) for q in q_candidates]

        chosen = None
        for thr in q_candidates:
            mask = base_mask & (dog >= thr)
            cc = np.argwhere(mask)
            if len(cc) == 0:
                continue
            # Do not flood the merge stage with millions of nearly zero peaks.
            chosen = (cc, dog[tuple(cc.T)])
            if len(cc) >= 20 or thr == q_candidates[-1]:
                break

        if chosen is not None:
            c, s = chosen
            candidates.append(c)
            scores.append(s.astype(np.float32, copy=False))

    if not candidates:
        return np.empty((0, 3), dtype=np.int32)

    coords = np.vstack(candidates).astype(np.int32, copy=False)
    score = np.concatenate(scores).astype(np.float32, copy=False)

    # Merge repeated detections from different scales.
    out = _physical_nms(
        coords,
        score,
        MIN_PEAK_SEPARATION_UM,
        MAX_DETECTIONS_PER_FRAME,
    )

    return out


def detect_movie(arr):
    T = int(arr.shape[0])
    detections = {}
    for t in range(T):
        det = detect_frame(read_frame(arr, t))
        detections[t] = det
        print(f"      frame {t:04d}: {len(det):5d} detections")
    return detections

# ---------------------------------------------------------------------------
# PHYSICAL GEOMETRY
# ---------------------------------------------------------------------------

def distance_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.empty((len(a), len(b)), dtype=np.float32)
    aa = np.asarray(a, dtype=np.float32) * SCALE
    bb = np.asarray(b, dtype=np.float32) * SCALE
    d = aa[:, None, :] - bb[None, :, :]
    return np.sqrt(np.sum(d * d, axis=2)).astype(np.float32)


def physical_distance(a, b) -> float:
    d = (np.asarray(a, dtype=np.float32) - np.asarray(b, dtype=np.float32)) * SCALE
    return float(np.sqrt(np.dot(d, d)))

# ---------------------------------------------------------------------------
# HUNGARIAN TRACKING
# ---------------------------------------------------------------------------

def assign_tracks(prev_xyz, curr_xyz, predicted=None):
    if len(prev_xyz) == 0 or len(curr_xyz) == 0:
        return []

    d_now = distance_matrix(prev_xyz, curr_xyz)
    if predicted is None:
        cost = d_now
    else:
        d_pred = distance_matrix(predicted, curr_xyz)
        cost = (1.0 - MOTION_WEIGHT) * d_now + MOTION_WEIGHT * d_pred

    gated = cost.copy()
    gated[gated > MAX_LINK_UM] = MAX_LINK_UM * 100.0
    r, c = linear_sum_assignment(gated)

    return [
        (int(i), int(j))
        for i, j in zip(r, c)
        if float(gated[i, j]) < MAX_LINK_UM
    ]


def build_tracks(detections):
    times = sorted(detections)
    if not times:
        return {}, []

    nodes = {}
    det_to_node = {}
    next_node = 1

    # Assign every detection one globally unique positive node ID.
    for t in times:
        det_to_node[t] = {}
        for ci, xyz in enumerate(detections[t]):
            nodes[next_node] = (
                int(t), int(xyz[0]), int(xyz[1]), int(xyz[2])
            )
            det_to_node[t][ci] = next_node
            next_node += 1

    edges = []
    active = {}

    # Seed tracks from the first timepoint.
    t0 = times[0]
    for ci, xyz in enumerate(detections[t0]):
        nid = det_to_node[t0][ci]
        active[nid] = {
            "last_t": t0,
            "last_xyz": xyz.astype(np.float32),
            "prev_xyz": None,
            "last_node": nid,
            "gap": 0,
        }

    for k in range(len(times) - 1):
        tp, tc = times[k], times[k + 1]
        curr_det = detections[tc]

        live_ids = [nid for nid, st in active.items() if st["last_t"] == tp]
        prev_xyz = np.asarray(
            [active[nid]["last_xyz"] for nid in live_ids], dtype=np.float32
        ) if live_ids else np.empty((0, 3), np.float32)

        predicted = None
        if live_ids:
            pred = []
            for nid in live_ids:
                st = active[nid]
                if st["prev_xyz"] is None:
                    p = st["last_xyz"].copy()
                else:
                    v = st["last_xyz"] - st["prev_xyz"]
                    # Clip predicted displacement in PHYSICAL space.
                    v_phys = v * SCALE
                    vm = float(np.sqrt(np.dot(v_phys, v_phys)))
                    if vm > VELOCITY_CLIP_UM_PER_FRAME:
                        v_phys *= VELOCITY_CLIP_UM_PER_FRAME / max(vm, 1e-6)
                        v = v_phys / SCALE
                    p = st["last_xyz"] + v
                pred.append(p)
            predicted = np.asarray(pred, dtype=np.float32)

        matches = assign_tracks(prev_xyz, curr_det, predicted)
        matched_live = set()
        matched_curr = set()

        for ri, ci in matches:
            nid = live_ids[ri]
            target = det_to_node[tc][ci]
            st = active[nid]
            edges.append((st["last_node"], target))
            matched_live.add(nid)
            matched_curr.add(ci)

            st["prev_xyz"] = st["last_xyz"].copy()
            st["last_xyz"] = curr_det[ci].copy()
            st["last_t"] = tc
            st["last_node"] = target
            st["gap"] = 0

        # Age unmatched tracks. Gap closure happens in a dedicated graph pass.
        for nid in live_ids:
            if nid in matched_live:
                continue
            st = active.get(nid)
            if st is None:
                continue
            st["gap"] += 1
            if st["gap"] > MAX_GAP:
                active.pop(nid, None)

        # Every unmatched detection begins a new track.
        for ci in range(len(curr_det)):
            if ci in matched_curr:
                continue
            nid = det_to_node[tc][ci]
            active[nid] = {
                "last_t": tc,
                "last_xyz": curr_det[ci].astype(np.float32),
                "prev_xyz": None,
                "last_node": nid,
                "gap": 0,
            }

    return nodes, edges

# ---------------------------------------------------------------------------
# GAP CLOSURE
# ---------------------------------------------------------------------------

def close_gaps(nodes, edges):
    outgoing = defaultdict(list)
    incoming = defaultdict(list)
    for s, t in edges:
        outgoing[s].append(t)
        incoming[t].append(s)

    ends = []
    starts = []
    for nid, (t, z, y, x) in nodes.items():
        xyz = np.asarray([z, y, x], dtype=np.float32)
        if not outgoing[nid]:
            ends.append((nid, int(t), xyz))
        if not incoming[nid]:
            starts.append((nid, int(t), xyz))

    ends.sort(key=lambda q: q[1])
    starts.sort(key=lambda q: q[1])

    used = set()
    recovered = []

    for en, et, ep in ends:
        best = None
        for sn, st, sp in starts:
            if sn in used:
                continue
            gap = st - et
            if gap < 2 or gap > MAX_GAP + 1:
                continue
            speed = physical_distance(ep, sp) / float(gap)
            if speed > MAX_LINK_UM:
                continue
            score = speed + 0.05 * abs(gap - 1)
            if best is None or score < best[0]:
                best = (score, sn)
        if best is not None:
            recovered.append((en, best[1]))
            used.add(best[1])

    return recovered

# ---------------------------------------------------------------------------
# DIVISION REPAIR
# ---------------------------------------------------------------------------

def detect_divisions(nodes, edges):
    if not ENABLE_DIVISIONS:
        return edges, 0

    existing = set(edges)
    outgoing = defaultdict(list)
    incoming = defaultdict(list)
    by_t = defaultdict(list)

    for nid, (t, z, y, x) in nodes.items():
        by_t[int(t)].append(nid)
    for s, t in edges:
        outgoing[s].append(t)
        incoming[t].append(s)

    added = 0

    for parent in sorted(nodes):
        children = list(outgoing[parent])
        if len(children) >= DIVISION_MAX_DAUGHTERS:
            continue

        pt, pz, py, px = nodes[parent]
        pxyz = np.asarray([pz, py, px], dtype=np.float32)

        # A division is formed only with eligible children at exactly t+1,
        # and a candidate child must not already have a different parent.
        cand = []
        for cid in by_t.get(int(pt) + 1, []):
            if cid in children:
                continue
            if incoming[cid]:
                continue
            ct, cz, cy, cx = nodes[cid]
            cxyz = np.asarray([cz, cy, cx], dtype=np.float32)
            d = physical_distance(pxyz, cxyz)
            if d <= DIVISION_MAX_UM:
                cand.append((d, cid))

        cand.sort()

        # Only add a second daughter to a parent that already has one child.
        if len(children) == 1 and cand:
            d1 = children[0]
            _, z1, y1, x1 = nodes[d1]
            xyz1 = np.asarray([z1, y1, x1], dtype=np.float32)
            chosen = None
            for d, cid in cand:
                _, z2, y2, x2 = nodes[cid]
                xyz2 = np.asarray([z2, y2, x2], dtype=np.float32)
                sep = physical_distance(xyz1, xyz2)
                if sep >= DIVISION_MIN_SEPARATION_UM:
                    chosen = (d, cid)
                    break
            if chosen is not None:
                edge = (parent, chosen[1])
                if edge not in existing:
                    edges.append(edge)
                    existing.add(edge)
                    incoming[chosen[1]].append(parent)
                    outgoing[parent].append(chosen[1])
                    added += 1

    return edges, added

# ---------------------------------------------------------------------------
# SUBMISSION CONSTRUCTION / VALIDATION
# ---------------------------------------------------------------------------

SUBMISSION_COLUMNS = [
    "id", "dataset", "row_type", "node_id", "t", "z", "y", "x",
    "source_id", "target_id",
]


def build_rows(dataset, nodes, edges, start_id):
    rows = []
    rid = int(start_id)

    for nid, (t, z, y, x) in sorted(nodes.items()):
        rows.append({
            "id": rid,
            "dataset": dataset,
            "row_type": "node",
            "node_id": int(nid),
            "t": int(t), "z": int(z), "y": int(y), "x": int(x),
            "source_id": -1, "target_id": -1,
        })
        rid += 1

    for s, t in edges:
        rows.append({
            "id": rid,
            "dataset": dataset,
            "row_type": "edge",
            "node_id": -1,
            "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": int(s), "target_id": int(t),
        })
        rid += 1

    return rows, rid


def validate_submission(df, expected_datasets):
    errors = []

    missing = [c for c in SUBMISSION_COLUMNS if c not in df.columns]
    if missing:
        errors.append(f"missing columns: {missing}")
        return errors

    if list(df.columns) != SUBMISSION_COLUMNS:
        errors.append("column order does not match required submission schema")

    if df.isnull().any().any():
        errors.append("null values found")

    if len(df):
        expected_ids = np.arange(len(df), dtype=np.int64)
        actual_ids = df["id"].to_numpy(dtype=np.int64)
        if not np.array_equal(actual_ids, expected_ids):
            errors.append("id is not consecutive from zero")

    found = set(df["dataset"].astype(str))
    missing_ds = sorted(set(expected_datasets) - found)
    if missing_ds:
        errors.append(f"missing datasets: {missing_ds}")

    nodes = df[df["row_type"] == "node"]
    edges = df[df["row_type"] == "edge"]

    if len(nodes) and (nodes["node_id"] < 1).any():
        errors.append("node_id < 1")

    node_ids = set(nodes["node_id"].astype(int))
    if len(edges):
        if (edges["source_id"] < 1).any():
            errors.append("edge source_id < 1")
        if (edges["target_id"] < 1).any():
            errors.append("edge target_id < 1")
        if (edges["source_id"] == edges["target_id"]).any():
            errors.append("self-loop")

        bad_refs = [
            (int(s), int(t)) for s, t in zip(edges.source_id, edges.target_id)
            if int(s) not in node_ids or int(t) not in node_ids
        ]
        if bad_refs:
            errors.append(f"edge references missing nodes: {bad_refs[:5]}")

        duplicate_edges = edges.duplicated(subset=["source_id", "target_id"]).any()
        if bool(duplicate_edges):
            errors.append("duplicate edges")

        # Temporal edge direction: source frame must precede target frame.
        nt = nodes.set_index("node_id")["t"].to_dict()
        wrong_time = [
            (int(s), int(t), int(nt[int(s)]), int(nt[int(t)]))
            for s, t in zip(edges.source_id, edges.target_id)
            if int(nt[int(s)]) >= int(nt[int(t)])
        ]
        if wrong_time:
            errors.append(f"non-forward edges: {wrong_time[:5]}")

        # A lineage fork is at most two daughters in this submission.
        out_counts = edges.groupby("source_id").size()
        if len(out_counts) and int(out_counts.max()) > 2:
            errors.append("node has more than two outgoing edges")

    return errors

# ---------------------------------------------------------------------------
# RUN ONE DATASET
# ---------------------------------------------------------------------------

def run_dataset(path):
    name = dataset_name(path)
    arr = make_array_adapter(open_zarr_array(path))

    print(f"    shape: {tuple(arr.shape)}")
    if len(arr.shape) != 4:
        raise ValueError(f"Adapter did not produce T,Z,Y,X: {arr.shape}")

    detections = detect_movie(arr)
    total_det = sum(len(v) for v in detections.values())

    if total_det == 0:
        # Schema-valid fallback only; normally a real dataset should detect.
        _, Z, Y, X = map(int, arr.shape)
        nodes = {1: (0, max(1, Z // 2), max(1, Y // 2), max(1, X // 2))}
        edges = []
        return nodes, edges, {
            "detections": 0,
            "nodes": 1,
            "edges": 0,
            "gap_edges": 0,
            "division_edges": 0,
            "fallback": True,
        }

    nodes, edges = build_tracks(detections)
    initial_edges = len(edges)

    gap_edges = close_gaps(nodes, edges)
    edges.extend(gap_edges)

    before_div = len(edges)
    edges, division_count = detect_divisions(nodes, edges)
    division_edges = len(edges) - before_div

    return nodes, edges, {
        "detections": int(total_det),
        "nodes": int(len(nodes)),
        "edges": int(len(edges)),
        "initial_edges": int(initial_edges),
        "gap_edges": int(len(gap_edges)),
        "division_edges": int(division_edges),
        "divisions": int(division_count),
        "fallback": False,
    }

# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------


def run():
    start = time.time()
    print("=" * 78)
    print("BIOHUB — ZEBRAFISH CELL TRACKING | COMPLETE OFFLINE PIPELINE")
    print("=" * 78)
    print(f"Physical scale: {tuple(float(v) for v in SCALE)} um/voxel")
    print(f"Link gate: {MAX_LINK_UM} um | Gap: {MAX_GAP} frames")

    root = find_competition_root()
    paths = find_test_datasets(root)
    names = [dataset_name(p) for p in paths]

    print(f"Input root: {root}")
    print(f"Datasets  : {len(paths)}")

    all_rows = []
    next_id = 0
    qc = {
        "input_root": str(root),
        "datasets": {},
        "config": {
            "scale_um_per_voxel": [float(v) for v in SCALE],
            "max_link_um": MAX_LINK_UM,
            "max_gap_frames": MAX_GAP,
            "division_max_um": DIVISION_MAX_UM,
            "division_enabled": ENABLE_DIVISIONS,
            "node_id_min": 1,
            "empty_value": -1,
        },
    }

    for i, path in enumerate(paths, 1):
        name = dataset_name(path)
        print("\n" + "-" * 78)
        print(f"[{i}/{len(paths)}] {name}")
        print("-" * 78)

        try:
            nodes, edges, stats = run_dataset(path)
            print(f"    graph: {len(nodes):,} nodes / {len(edges):,} edges")
            print(f"    gap edges: {stats.get('gap_edges', 0):,}")
            print(f"    division edges: {stats.get('division_edges', 0):,}")

            rows, next_id = build_rows(name, nodes, edges, next_id)
            all_rows.extend(rows)
            qc["datasets"][name] = stats

        except Exception as exc:
            print(f"    FAILED: {type(exc).__name__}: {exc}")
            # Preserve a valid row for the dataset so every test dataset is
            # represented. The QC report makes the failure explicit.
            all_rows.append({
                "id": next_id,
                "dataset": name,
                "row_type": "node",
                "node_id": 1,
                "t": 0, "z": 1, "y": 1, "x": 1,
                "source_id": -1, "target_id": -1,
            })
            next_id += 1
            qc["datasets"][name] = {
                "fallback": True,
                "error_type": type(exc).__name__,
                "error": str(exc),
            }

    if not all_rows:
        raise RuntimeError("No output rows were generated")

    df = pd.DataFrame(all_rows, columns=SUBMISSION_COLUMNS)
    df["id"] = np.arange(len(df), dtype=np.int64)

    int_cols = [
        "id", "node_id", "t", "z", "y", "x", "source_id", "target_id"
    ]
    for c in int_cols:
        df[c] = df[c].astype(np.int64)

    errors = validate_submission(df, names)

    print("\n" + "=" * 78)
    print("VALIDATION")
    print("=" * 78)
    if errors:
        for e in errors:
            print("ERROR:", e)
        raise RuntimeError("Submission validation failed")

    print("PASSED")
    print(f"rows : {len(df):,}")
    print(f"nodes: {(df['row_type'] == 'node').sum():,}")
    print(f"edges: {(df['row_type'] == 'edge').sum():,}")

    out_csv = OUT_DIR / "submission.csv"
    df.to_csv(out_csv, index=False)

    qc["rows"] = int(len(df))
    qc["nodes"] = int((df["row_type"] == "node").sum())
    qc["edges"] = int((df["row_type"] == "edge").sum())
    qc["runtime_seconds"] = float(time.time() - start)
    qc["submission_csv"] = str(out_csv)

    if SAVE_QC_JSON:
        with open(OUT_DIR / "tracking_qc.json", "w", encoding="utf-8") as f:
            json.dump(qc, f, indent=2)

    print(f"\nSAVED: {out_csv}")
    if SAVE_QC_JSON:
        print(f"QC   : {OUT_DIR / 'tracking_qc.json'}")
    print(f"TIME : {time.time() - start:.1f} s")
    print("\nPreview:")
    print(df.head(12).to_string(index=False))
    print("\nDONE — submit submission.csv from the Kaggle Output tab.")
    return df


if __name__ == "__main__":
    submission = run()


BIOHUB — ZEBRAFISH CELL TRACKING | COMPLETE OFFLINE PIPELINE
Physical scale: (1.625, 0.40625, 0.40625) um/voxel
Link gate: 7.0 um | Gap: 2 frames
Input root: /kaggle/input/competitions/biohub-cell-tracking-during-development
Datasets  : 4

------------------------------------------------------------------------------
[1/4] 44b6_0113de3b
------------------------------------------------------------------------------
    FAILED: RuntimeError: zarr is not installed in this Kaggle environment. Attach/use a Kaggle runtime that includes zarr before running the notebook.

------------------------------------------------------------------------------
[2/4] 44b6_0b24845f
------------------------------------------------------------------------------
    FAILED: RuntimeError: zarr is not installed in this Kaggle environment. Attach/use a Kaggle runtime that includes zarr before running the notebook.

------------------------------------------------------------------------------
[3/4] 6bba_05b6850b